# **응급상황 자동 인식 및 응급실 연계 서비스**
# 단계2 : 파인튜닝!

## **0.미션**

### (1) 미션1
단계 2에서는 사전 학습된 한국어 BERT 모델을 다운받아, 파인튜닝을 시도합니다. 이를 위해 먼저 적절한 데이터를 수집해야 합니다.

* 학습 데이터 준비
    * 등급별 50개씩 샘플 데이터를 제공합니다.
    * **응급상황 등급(1~5) 별 최소 100건씩** 데이터를 추가로 수집(제작)하세요.
        * 중증도 카테고리 파일에 있는 키워드 참조
        * 제작시 GPT를 적극 활용해 봅시다.
    * 총 데이터 크기 : 600건
        * 제공데이터 100건(20건씩 5개 등급)
        * 추가 제작 500건(100건씩 5개 등급)

|text|label|
|----|----|
|GPT모델의 출력 형태. 응급상황 내용 | 응급등급(1~5)|



### (2) 미션2
미션1에서 준비한 학습데이터를 활용해서 파인튜닝을 시도합니다.

* 사용할 모델 : **klue/bert-base**
* 성능을 높이기 위해 조정할 만한 점(아래 수치는 가이드)
    - 데이터 양 늘리기
    - 하이퍼 파라미터
        - epoch 수 : 5 ~ 20
        - learning_rate : 1e-5 ~ 1e-2
        - batch_size : 16 ~ 32
        - weight_decay : 0.01 ~ 0.1

## 1.환경준비

* 세부 요구사항
    - 경로 설정 : 구글콜랩
        * 구글 드라이브 바로 밑에 project6 폴더를 만들고,
        * 제공 받은 파일을 복사해 넣습니다.
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
        * 필요하다고 판단되는 라이브러리를 추가하세요.


### (1) 라이브러리 설치 및 로딩

In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


* (필요 시) 설치 후 세션 다시 시작

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

* 함수 생성
    * evaluate : 모델 생성후, 데이터셋(예 : 검증셋)을 이용해서 모델 성능을 평가하기 위한 용도
    * predict :  하나의 문장을 입력받아, 예측결과를 반환

In [3]:
# 검증셋 평가 함수
def evaluate(val_ds, model, device, tokenizer):
    # 입력 데이터셋 토크나이징 (attention_mask 포함)
    inputs = tokenizer(val_ds['text'], return_tensors="pt", padding=True,
                       truncation=True, max_length=128
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}  # 입력 텐서를 동일한 디바이스로 이동

    # 모델을 지정된 디바이스로 이동
    model = model.to(device)

    with torch.no_grad():  # 평가 과정에서 기울기 계산 비활성화
        outputs = model(**inputs)  # attention_mask를 포함해 입력

    # 예측 및 확률 계산
    probabilities = outputs.logits.softmax(dim=1)

    # probabilities가 GPU에 있을 경우에만 CPU로 이동
    if probabilities.is_cuda:
        probabilities = probabilities.cpu().detach().numpy()
    else:
        probabilities = probabilities.detach().numpy()

    pred = np.argmax(probabilities, axis=1)

    # GPU 메모리에서 필요 없는 텐서 제거 및 캐시 정리
    del inputs
    torch.cuda.empty_cache()

    return pred, probabilities

In [4]:
# 데이터 예측 함수
def predict(text, model, tokenizer):
    # 입력 문장 토크나이징
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  # 각 텐서를 GPU로 이동

    # 모델 예측
    with torch.no_grad():
        outputs = model(**inputs)

    # 로짓을 소프트맥스로 변환하여 확률 계산
    logits = outputs.logits
    probabilities = logits.softmax(dim=1)

    # 가장 높은 확률을 가진 클래스 선택
    pred = torch.argmax(probabilities, dim=-1).item()

    return pred, probabilities

### (2) 구글드라이브 연결

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
path = '/content/drive/MyDrive/mini-project-6-1/'

### (3) 데이터 로딩
* 이전 미션에서 준비한 데이터셋을 로딩합니다.

In [7]:
df = pd.read_csv(path+'dataset.csv')

In [8]:
df2 = pd.read_csv(path+'ktas.csv', index_col=0)

In [9]:
df = pd.concat([df, df2], ignore_index=True)
df

,text,label
0,환자가 심각한 호흡곤란으로 숨을 쉬기 어려워하며 피부가 푸르스름하게 변색된 상태,1
1,환자가 얼굴이 창백하고 식은땀을 흘리며 맥박이 거의 잡히지 않는 쇼크 상태,1
2,"환자가 바닥에 쓰러져 의식을 잃었고, 주변의 소리에 반응하지 않으며, 몸이 움직이지...",1
3,환자가 심한 호흡곤란으로 숨을 쉬지 못해 곤란해하는 상태,1
4,환자가 심각한 쇼크로 인해 혈압이 떨어지고 의식을 잃을 위험에 있는 상태,1
...,...,...
595,환자가 기침과 함께 가벼운 두통을 겪는 상태,4
596,환자가 만성적인 두통과 함께 체온 상승을 보이는 상태,4
597,환자가 경미한 발열을 동반하는 통증을 호소하는 상태,4
598,환자가 약간의 구토를 보이며 복통을 겪고 있는 상태,4


* y 분포 확인하기

In [10]:
df['label'].value_counts()

,count
label,
1,120
2,120
3,120
4,120
5,120


* label 에 대한 정수 인코딩
    * 정수 인코딩은 0 부터 시작해야 함.

In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
df

,text,label
0,환자가 심각한 호흡곤란으로 숨을 쉬기 어려워하며 피부가 푸르스름하게 변색된 상태,0
1,환자가 얼굴이 창백하고 식은땀을 흘리며 맥박이 거의 잡히지 않는 쇼크 상태,0
2,"환자가 바닥에 쓰러져 의식을 잃었고, 주변의 소리에 반응하지 않으며, 몸이 움직이지...",0
3,환자가 심한 호흡곤란으로 숨을 쉬지 못해 곤란해하는 상태,0
4,환자가 심각한 쇼크로 인해 혈압이 떨어지고 의식을 잃을 위험에 있는 상태,0
...,...,...
595,환자가 기침과 함께 가벼운 두통을 겪는 상태,3
596,환자가 만성적인 두통과 함께 체온 상승을 보이는 상태,3
597,환자가 경미한 발열을 동반하는 통증을 호소하는 상태,3
598,환자가 약간의 구토를 보이며 복통을 겪고 있는 상태,3


### (4) GPU 설정
* 파이토치를 위한 설정

In [12]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 2.데이터 준비

### (1) Dataset 만들기 : train, val

In [13]:
# 데이터셋 분할
train, val = train_test_split(df, test_size=0.2, random_state=42)

In [14]:
train[:3]

,text,label
145,고열로 인해 환자가 의식을 잃고 경련을 일으키며 쓰러진 상태 ...,0
9,환자가 경련을 일으키며 몸이 떨리고 혼란스러운 상태,0
375,"환자가 가벼운 기침과 미열을 동반한 감기 증상을 보이며, 기운이 없는 상태 ...",2


In [15]:
# 텐서 데이터셋
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)

In [16]:
train_ds[:3]

{'text': ['고열로 인해 환자가 의식을 잃고 경련을 일으키며 쓰러진 상태                                                         ',
  '환자가 경련을 일으키며 몸이 떨리고 혼란스러운 상태',
  '환자가 가벼운 기침과 미열을 동반한 감기 증상을 보이며, 기운이 없는 상태                                              '],
 'label': [0, 0, 2],
 '__index_level_0__': [145, 9, 375]}

### (2) 토큰화

* 사용할 모델("klue/bert-base")의 토크나이저를 저장

In [23]:
# 모델과 토크나이저 불러오기
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [24]:
# 토큰화 함수 생성 및 작업
def preprocess_function(data):
    return tokenizer(data['text'], truncation=True, padding=True)

train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

## 3.Fine-Tuning

### (1) 사전학습 모델 준비
* 사용할 모델("klue/bert-base")의 토크나이저를 저장
* 이때, 출력층 노드 수를 지정합니다.

In [36]:
# 모델 설정 (다중 분류를 위해 num_labels 지정)
n = 5
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = n).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### (2) 학습
* 절차
    * TrainArguments 설정
        * 조정할 하이퍼파라미터 : learning_rate, batch_size, epochs, weight_decay 등
        * 나머지 옵션은 손 대지 맙시다.
    * Trainer 설정
    * 학습
* 성능 : 데이터, 하이퍼파라미터에 의해 결정

In [37]:
# TrainingArguments

lr = 2e-6                # <--
batch_size = 16       # <--
epochs = 20            # <--

training_args = TrainingArguments(
    output_dir = './results',          # 출력 디렉토리
    eval_strategy = "epoch",            # 에폭마다 평가
    save_strategy = "epoch",           # 에폭마다 체크포인트 저장
    learning_rate = lr,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    num_train_epochs  = epochs,
    weight_decay = 0.02,               # <--[조정가능]weight decay
    load_best_model_at_end = True,     # 가장 좋은 모델을 마지막에 로드
    logging_dir ='./logs',            # 로깅 디렉토리
    logging_steps = 10,                # 로깅 스텝
    report_to="tensorboard"          # TensorBoard에 로깅
)

In [38]:
# Trainer 설정
patience = 5       # <--

trainer = Trainer(
    model=model,                         # 학습할 모델
    args=training_args,                  # TrainingArguments
    train_dataset = train_ds,
    eval_dataset = val_ds,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)], # 조기 종료
)

In [ ]:
# 모델 학습
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.545000,1.567623
2,1.462300,1.462677
3,1.339200,1.376771
4,1.277400,1.310839
5,1.195700,1.233448
6,1.095700,1.170366
7,1.049100,1.124712
8,0.971700,1.069029
9,0.907100,1.026791
10,0.872400,1.003872


### (3) 모델 검증평가
* evaluate 함수를 이용해서 검증셋에 대한 성능 평가를 수행합니다.

In [ ]:
# 모델 평가
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
# 모델 평가
eval_results2 = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Epoch,Training Loss,Validation Loss
1,1.608100,1.523396
2,1.431000,1.399081
3,1.334700,1.257569
4,1.156800,1.128096
5,1.032300,1.054283
6,0.907500,0.964025
7,0.849200,0.892590
8,0.725100,0.840355
9,0.659100,0.801568
10,0.587300,0.764094


Evaluation results: {'eval_loss': 0.7231717705726624}


In [ ]:
pred, prob = evaluate(val_ds, model, device, tokenizer)

In [ ]:
target_names = [str(cls) for cls in le.classes_]

print(confusion_matrix(val_ds['label'], pred))
print(classification_report(val_ds['label'], pred, target_names = target_names))

[[19  4  0  0  0]
 [ 4 21  1  0  0]
 [ 0  4 16  1  1]
 [ 0  3  4 16  7]
 [ 0  0  0  4 15]]
              precision    recall  f1-score   support

           1       0.83      0.83      0.83        23
           2       0.66      0.81      0.72        26
           3       0.76      0.73      0.74        22
           4       0.76      0.53      0.63        30
           5       0.65      0.79      0.71        19

    accuracy                           0.72       120
   macro avg       0.73      0.74      0.73       120
weighted avg       0.73      0.72      0.72       120



### (4) 모델 사용
* 응급상황에 대한 샘플 문자을 넣고 등급을 적절히 분류하는지 눈으로 확인해 봅시다.

In [ ]:
text = "환자가 급성 저혈당으로 의식이 혼미한 상태"
predicted_class, probabilities = predict(text, model, tokenizer)

print(f"예측된 클래스: {predicted_class}")
print(f"예측된 클래스 이름: {le.inverse_transform([predicted_class])}")
print(f"클래스별 확률: {probabilities}")

예측된 클래스: 0
예측된 클래스 이름: [1]
클래스별 확률: tensor([[0.9542, 0.0123, 0.0092, 0.0167, 0.0076]], device='cuda:0')


## 4.모델 저장

### (1) 모델 저장
* save_pretrained 메서드를 사용해 모델과 토크나이저를 디렉토리에 저장합니다.
* 저장 대상 : 모델, 토크나이저

In [ ]:
save_directory = path + "fine_tuned_bert/1"

In [ ]:
# 모델과 토크나이저 저장 경로 : 폴더가 없으면, fine_tuned_bert를 생성
save_directory = path + "fine_tuned_bert/1"

# 모델 저장
model.save_pretrained(save_directory)

# 토크나이저도 함께 저장
tokenizer.save_pretrained(save_directory)

### (2) 모델 로딩 및 사용
* 저장한 모델을 나중에 다시 불러와 예측이나 추가 작업에 사용할 수 있습니다. Hugging Face from_pretrained 메서드를 사용하여 저장 경로에서 모델과 토크나이저를 로드합니다.
* 로딩 대상 : 모델, 토크나이저

In [ ]:
# 모델 로드
model2 = AutoModelForSequenceClassification.from_pretrained(save_directory)

# 토크나이저 로드
tokenizer2 = AutoTokenizer.from_pretrained(save_directory)

In [ ]:
text = "넘어졌는데, 지금 출혈이 있고, 머리가 어지럽습니다."
predicted_class, probabilities = predict(text, model2, tokenizer2)

print(f"예측된 클래스: {predicted_class}")
print(f"예측된 클래스 이름: {predicted_class+1}등급")
print(f"클래스별 확률: {probabilities}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [ ]:
df_test = pd.read_csv(path + 'dataset100.csv')

le_test = LabelEncoder()
df_test['label'] = le_test.fit_transform(df_test['label'])

ds_test = Dataset.from_pandas(df_test)

In [ ]:
df_test

,text,label
0,환자가 심각한 호흡곤란으로 숨을 쉬기 어려워하며 피부가 푸르스름하게 변색된 상태,0
1,환자가 얼굴이 창백하고 식은땀을 흘리며 맥박이 거의 잡히지 않는 쇼크 상태,0
2,"환자가 바닥에 쓰러져 의식을 잃었고, 주변의 소리에 반응하지 않으며, 몸이 움직이지...",0
3,환자가 심한 호흡곤란으로 숨을 쉬지 못해 곤란해하는 상태,0
4,환자가 심각한 쇼크로 인해 혈압이 떨어지고 의식을 잃을 위험에 있는 상태,0
...,...,...
495,환자가 약간의 손목 통증을 느끼지만 걸을 때 불편함이 없는 상태,4
496,환자가 경미한 발열로 약간의 불편함을 느끼는 상태,4
497,환자가 가벼운 눈의 자극으로 불편함을 느끼는 상태,4
498,환자가 손톱에 작은 찔림 상처를 느끼는 상태,4


In [ ]:
ds_test[:3]

{'text': ['환자가 심각한 호흡곤란으로 숨을 쉬기 어려워하며 피부가 푸르스름하게 변색된 상태',
  '환자가 얼굴이 창백하고 식은땀을 흘리며 맥박이 거의 잡히지 않는 쇼크 상태',
  '환자가 바닥에 쓰러져 의식을 잃었고, 주변의 소리에 반응하지 않으며, 몸이 움직이지 않는 상태'],
 'label': [0, 0, 0]}

In [ ]:
pred2, prob2 = evaluate(ds_test, model2, device, tokenizer2)

In [ ]:
target_names = [str(cls) for cls in le.classes_]

print(confusion_matrix(ds_test['label'], pred2))
print(classification_report(ds_test['label'], pred2, target_names = target_names))

[[82 16  0  1  1]
 [ 9 36 17 25 13]
 [ 1  2 37 19 41]
 [ 1  2 13 22 62]
 [ 0  0  1  9 90]]
              precision    recall  f1-score   support

           1       0.88      0.82      0.85       100
           2       0.64      0.36      0.46       100
           3       0.54      0.37      0.44       100
           4       0.29      0.22      0.25       100
           5       0.43      0.90      0.59       100

    accuracy                           0.53       500
   macro avg       0.56      0.53      0.52       500
weighted avg       0.56      0.53      0.52       500



## **Mission Complete!**

수고 많았습니다!

In [ ]:
import shutil

shutil.rmtree('results', 'logs')